# Analyze Summit DisruptCNN artifacts

Analyze existing **checkpoints**, **splits**, and **test performance** outputs from a collaborator's run directory (e.g. `/pscratch/sd/r/rchurchi/d3d_ecei/summit_disruptcnn`).

Set `ARTIFACTS_ROOT` below to the path where you copied the directory (or a symlink). The notebook will:
1. Scan for `checkpoint.*.pth.tar`, `model_best.*.pth.tar`, `splits.*.npz`, and `test_disrupt.*`
2. Parse job IDs and build a summary table
3. Inspect checkpoint contents (keys, epoch, optional state_dict keys)
4. Inspect splits (train/val/test indices)
5. List and optionally load test performance outputs (text or metrics)

In [ ]:
import re
from pathlib import Path
import numpy as np

# Point to the summit_disruptcnn directory (copy or symlink locally)
ARTIFACTS_ROOT = Path("/pscratch/sd/r/rchurchi/d3d_ecei/summit_disruptcnn")  # or local path
if not ARTIFACTS_ROOT.exists():
    print(f"Not found: {ARTIFACTS_ROOT}")
    print("Set ARTIFACTS_ROOT to your local copy or mounted path.")
else:
    print(f"Root: {ARTIFACTS_ROOT}")
    print(f"Exists: {ARTIFACTS_ROOT.exists()}")

## 1. Scan and categorize files

In [ ]:
def extract_id(name: str) -> str:
    """Extract numeric job/run ID (e.g. checkpoint.236423.pth.tar -> 236423)."""
    m = re.search(r"\.(\d+)(?:\.|\.pth|\.npz)", name)
    return m.group(1) if m else ""

checkpoints = []
model_bests = []
splits = []
test_dirs = []

if ARTIFACTS_ROOT.exists():
    for f in sorted(ARTIFACTS_ROOT.iterdir()):
        name = f.name
        if name.startswith("checkpoint.") and ".pth.tar" in name:
            jid = extract_id(name)
            epoch_m = re.search(r"epoch\.(\d+)", name)
            epoch = int(epoch_m.group(1)) if epoch_m else None
            checkpoints.append((f, jid, epoch))
        elif name.startswith("model_best.") and name.endswith(".pth.tar"):
            model_bests.append((f, extract_id(name)))
        elif name.startswith("splits.") and name.endswith(".npz"):
            splits.append((f, extract_id(name)))
        elif name.startswith("test_disrupt.") and f.is_dir():
            test_dirs.append((f, extract_id(name)))

print(f"Checkpoints: {len(checkpoints)}, model_best: {len(model_bests)}, splits: {len(splits)}, test_disrupt dirs: {len(test_dirs)}")
if model_bests:
    print("Sample model_best:", model_bests[0][0].name)
if splits:
    print("Sample splits:", splits[0][0].name)

In [ ]:
# Optional: list job IDs that have model_best
if model_bests:
    jids = sorted({jid for _, jid in model_bests}, key=int)
    print(f"Job IDs with model_best: {len(jids)}. First 10: {jids[:10]}")

## 2. Summary table (job IDs with artifacts)

In [ ]:
mb_ids = {jid for _, jid in model_bests}
split_ids = {jid for _, jid in splits}
test_ids = {jid for _, jid in test_dirs}
all_ids = sorted(mb_ids | split_ids | test_ids, key=int)

rows = []
for jid in all_ids[:80]:
    ckpts = [c for c in checkpoints if c[1] == jid]
    epochs = sorted({c[2] for c in ckpts if c[2] is not None})
    rows.append({
        "job_id": jid,
        "model_best": jid in mb_ids,
        "splits": jid in split_ids,
        "test_disrupt": jid in test_ids,
        "n_ckpts": len(ckpts),
        "epochs_sample": epochs[:5] if epochs else [],
    })

try:
    import pandas as pd
    df = pd.DataFrame(rows)
    display(df)
except Exception:
    for r in rows[:20]:
        print(r)

## 3. Inspect one checkpoint

In [ ]:
import torch

if not model_bests:
    print("No model_best files.")
else:
    ckpt_path = model_bests[0][0]
    print(f"Loading: {ckpt_path.name}")
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    print("Keys:", list(ckpt.keys()))
    for k in ["epoch", "best_f1", "state_dict"]:
        if k in ckpt:
            v = ckpt[k]
            if k == "state_dict":
                print(f"  {k}: {len(v)} items")
            else:
                print(f"  {k}: {v}")

## 4. Inspect one splits file

In [ ]:
if not splits:
    print("No splits files.")
else:
    sp_path = splits[0][0]
    print(f"Loading: {sp_path.name}")
    data = np.load(sp_path, allow_pickle=True)
    print("Keys:", list(data.keys()))
    for k in data.files:
        arr = data[k]
        print(f"  {k}: shape={getattr(arr, 'shape', 'N/A')}, dtype={getattr(arr, 'dtype', 'N/A')}")

## 5. Inspect test_disrupt output (text performances)

In [ ]:
if not test_dirs:
    print("No test_disrupt dirs.")
else:
    td_path = test_dirs[0][0]
    print(f"Contents of {td_path.name}/:")
    for f in sorted(td_path.iterdir()):
        print(f"  {f.name}")
    # If there are .txt or .out files, show first few lines of one
    txts = list(td_path.glob("*.txt")) + list(td_path.glob("*.out")) + list(td_path.glob("*.log"))
    if txts:
        with open(txts[0]) as f:
            print("\nFirst 30 lines of", txts[0].name)
            for i, line in enumerate(f):
                if i >= 30: break
                print(line.rstrip())